In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

import sys
from pathlib import Path

In [2]:
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ufc_betting").is_dir())
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))
from ufc_betting.BettingStrategy.MonteCarlo.mc_engine import simulate
from ufc_betting.BettingStrategy.MonteCarlo.mc_plots import plot_final_distributions, plot_negative_fraction_histograms, mc_analysis
from ufc_betting.config import settings


In [ ]:
# Raw feature rows are needed to recompute probabilities and allocations.
fights = pd.read_csv(settings.data_dir / "model_predictions" / "test_logit_data_open_with_open.csv")
fights = fights.loc[fights.winner.isin([0, 1])].copy()  # Joint decay currently supports decisive outcomes.
# The saved model frame omits title_fight. Attach actual event-level title metadata.
stats = pd.concat([pd.read_csv(settings.stats_history_file), pd.read_csv(settings.non_merged_stats_file)])
title_events = stats.groupby("event_date").title_fight.max()
fights["event_has_title"] = fights.date.map(title_events)
if fights.event_has_title.isna().any():
    raise ValueError("Missing title-event metadata for some simulation dates")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\jcmar\\my_files\\SportsBetting\\Data\\model_results\\test_set_data_open_with_open.csv'

In [4]:
from ufc_betting.Models.RiskManagement.lead_time_backtest import load_lead_time_params

odds_type = "intermediate"  # "open", "close1", "close2", or "intermediate"
betting_control = "close1"  # Fixed controls; independently selectable from prices
fixed_params = load_lead_time_params(settings.data_dir)
p = fixed_params[betting_control]
kelly_kwargs = dict(
    max_drawdown=p["mdd"], N=p["N"], parlay_mdd=p["mdd_parlay"], N_parlay=p["N_parlay"],
    z=p["z"], moneyline_american_bounds=(p["moneyline_min_american"], p["moneyline_max_american"]),
    calc_parlay=True, adjust_mdd_by_edge=False,
    # Remove these two arguments to use ordinary fixed-policy Kelly sizing.
    decay_rates={"k_down": 49.91702161206258, "k_up": 0.5504910146566548,
                 "k_parlay_down": 0.5760956787848726, "k_parlay_up": 0.18014878632477732},
    opening_params=fixed_params["open"],
)


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\jcmar\\my_files\\SportsBetting\\Data\\search_results\\open\\betting_data.csv'

In [ ]:
# Price endpoint U is uniform between each fight's close1 and close2 quotes.
# Intermediate prices are then sampled from the conditional density (not uniformly).
# The density fit is descriptive on the archive's test cohort; not held-out validation.
mc_ml_open, mc_parlay_open, mc_total_open = simulate(
    fights,
    odds_type=odds_type,
    kelly_kwargs=kelly_kwargs,
    init_bankroll=500,
    n_sims=100,
    len_temp=1,  # Settle/reinvest after each event; larger batches can exceed bankroll
    days_out=5,
    random_state=42,
    title_col="event_has_title",
)


In [ ]:
parlay_test = mc_parlay_open[-1]
print(len(parlay_test))
parlay_win = (parlay_test > 0).sum() / (len(parlay_test) )
print(parlay_win)

ml_test = mc_ml_open[0]
print(len(ml_test))
ml_win = (ml_test > 0).sum() / len(ml_test)
print(ml_win)

total_test = np.diff(mc_total_open[-1])
total_win = (total_test > 0).sum() / len(total_test) 
print(total_win)
print(len(total_test))

In [ ]:
path1 = r"C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\mc_paths_delay.png"
path2 = r"C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\mc_hists_delay.png"
path3 = r"C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\mc_below_zero_delay.png"


mc_analysis(mc_ml_open, mc_parlay_open, mc_total_open, path1)
plot_final_distributions(mc_ml_open, mc_parlay_open, mc_total_open, path2)
plot_negative_fraction_histograms(mc_ml_open, mc_parlay_open, mc_total_open, path3)


In [ ]:
final_distributions = np.array(mc_total_open)[:, -1]
benchmarks = np.array([10000, 50000, 100000, 250000, 500000, 1000000])

df_benchmark = []
for bm in benchmarks:
    prob_above_bm = (final_distributions > bm).mean()
    df_benchmark.append({'Benchmark': bm, 'Probability Above': prob_above_bm})

df_benchmark = pd.DataFrame(df_benchmark)
df_benchmark.to_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\benchmark_results.csv', index=False)
df_benchmark.head()


In [ ]:
path1 = r"C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\mc_paths.png"
mc_analysis(mc_ml_open, mc_parlay_open, mc_total_open, path1)

path2 = r"C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\mc_hists.png"
plot_final_distributions(mc_ml_open, mc_parlay_open, mc_total_open, path2)

path3 = r"C:\Users\jcmar\my_files\SportsBetting\Data\plot_pngs\mc_below_zero.png"
plot_negative_fraction_histograms(mc_ml_open, mc_parlay_open, mc_total_open, path3)

